In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Lesson 2: Understanding JIT and Compilation

## Overview

`jax.jit` is the single biggest reason JAX-on-GPU code is fast — and the single biggest source of "why is this so slow?" questions when something goes wrong. The first time you call a JIT-compiled function it can take seconds; the second time it takes microseconds. Change the input shape and JAX might recompile from scratch. Put an ordinary Python `if` on a JAX value and you usually get a `TracerBoolConversionError`.

None of this is mysterious once you understand what `jit` actually does. JAX **traces** your Python function with abstract placeholders, hands the recorded program to **XLA**, and caches the compiled executable. For everyday use, think of the cache key as the jitted function plus the input structure, array shapes, dtypes, and any static argument values. Backend/device details and compiler options can matter too, but shapes, dtypes, and static values are the parts you usually control. This lesson walks through that mental model end to end so you can write JIT code that compiles once and runs fast forever.

**What you'll do:**

* See **tracing** in action by watching a Python `print` fire only on the first call
* Measure the cost of **compilation** vs. the cost of **execution**
* Understand the **compile cache** and what kinds of input changes trigger a recompile
* Hit (and fix) the most common pitfall: Python control flow on a traced JAX value
* Use **`jax.lax`** primitives and **padding** to keep shapes stable
* Mark Python-side constants with **`static_argnums`** when you actually want them baked in
* Inspect a traced function with **`jax.make_jaxpr`** to see what XLA will compile

## Hardware and software requirements

This notebook runs in the same environment as Lesson 1, a compatible NVIDIA driver, and a CUDA-enabled JAX install. Use the [nvcr.io/nvidia/jax](https://catalog.ngc.nvidia.com/orgs/nvidia/containers/jax) container or install a current JAX GPU wheel, for example `pip install --upgrade "jax[cuda13]"` for the recommended CUDA 13 path or `pip install --upgrade "jax[cuda12]"` for CUDA 12 compatibility.

Keep the driver requirements from Lesson 1 in mind: CUDA 13 wheels require NVIDIA driver 580+ on Linux, while CUDA 12 wheels require driver 525+.

## Setup

Let's import JAX, NumPy, and a few standard-library helpers, then confirm we're on a GPU. If the assertion below fails, jump back to the first lesson and re-run the setup section.

In [ ]:
import time
from functools import partial

import jax
import jax.numpy as jnp

devices = jax.devices()
gpu_devices = [d for d in devices if d.platform == "gpu"]

print(f"JAX version:     {jax.__version__}")
print(f"Default backend: {jax.default_backend()}")
print(f"Devices:         {devices}")

assert gpu_devices, f"This lesson assumes a GPU backend. Available devices: {devices}"
print(f"GPU devices:     {gpu_devices}")

## What `jax.jit` actually does

When you call a plain (non-JIT) JAX function, each operation runs through Python and dispatches to the GPU as it executes. `jax.jit` changes that. Instead of running your function with real arrays, it **traces** the function: JAX calls it once with abstract placeholders (one per argument, carrying only the shape and dtype) and records every JAX operation you perform on those placeholders into an intermediate representation called a **jaxpr**.

JAX lowers the jaxpr to StableHLO, hands that lowered program to **XLA**, and XLA compiles an optimized executable for the target device. XLA may fuse operations, but a compiled function can still lower to multiple GPU kernels. From then on, calling the function jumps straight to the cached executable.

**The three phases of a JIT call**

<div style="font-family: Arial, sans-serif; max-width: 980px; line-height: 1.35; display: grid; grid-template-columns: 1fr 28px 1fr 28px 1fr; gap: 10px; align-items: stretch; margin-top: 12px;">
  <div style="grid-column: 1; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">1. Trace</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Record the computation</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">Python runs once; JAX records every operation on abstract placeholders into a <code>jaxpr</code>.</div>
  </div>
  <div style="grid-column: 2; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #57606a;">&rarr;</div>
  <div style="grid-column: 3; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">2. Compile</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Lower to a GPU executable</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">JAX lowers the <code>jaxpr</code> to StableHLO and XLA compiles an optimized executable for the device.</div>
  </div>
  <div style="grid-column: 4; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #57606a;">&rarr;</div>
  <div style="grid-column: 5; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">3. Execute</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Reuse the cached executable</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">Every later call with matching shapes and dtypes skips trace/compile and runs the cached program.</div>
  </div>
</div>
<br>

Step 1 is why a Python `print` inside a JIT function only fires on the first call. Steps 1 and 2 together are why the first call is slow. Step 3 is why every call after is fast.

### Tracing in action

Let's prove that the function body only runs once per input signature. We'll put a Python-level `print` inside the function — it executes during tracing, but it's *not* part of the compiled GPU program, so subsequent calls with the same shape and dtype don't print anything.

In [ ]:
@jax.jit
def f(x):
    """Jitted demo function that prints during tracing so we can see exactly when JAX retraces."""
    # This print runs during tracing only - not on every GPU execution
    print(f"  tracing with shape={x.shape} dtype={x.dtype}")
    return x ** 2 + 1


print("Call 1 (new shape):")
_ = f(jnp.arange(4, dtype=jnp.float32)).block_until_ready()

print("Call 2 (same shape):")
_ = f(jnp.arange(4, dtype=jnp.float32)).block_until_ready()

print("Call 3 (new shape):")
_ = f(jnp.arange(5, dtype=jnp.float32)).block_until_ready()

The print fires on call 1 (first time JAX sees shape `(4,)` and dtype `float32`) and on call 3 (first time it sees shape `(5,)`). On call 2, JAX finds an existing compiled executable for `(4,) float32` and skips both tracing and compilation entirely.

> **Important:** Anything that depends on a Python-side side effect — a `print`, a logging call, mutating an external counter — only happens during tracing. Don't put logic you need on every call inside a JIT function expecting it to fire every call. For actual runtime logging from inside a JIT, JAX provides `jax.debug.print`, which is wired into the compiled program.

### Why the first call is slow

The "first time you see a new signature" cost is real, and it's where most "JAX is mysteriously slow" reports come from. Let's measure how much of the first call is compilation, and how much is execution.

In [ ]:
def heavy(x):
    """20 chained nonlinearities so the first-call compilation is visibly more expensive than the cached execution."""
    y = x
    for _ in range(20):
        y = jnp.sin(y) * jnp.cos(y) + jnp.tanh(y)
    return y


heavy_jit = jax.jit(heavy)
x = jnp.arange(1_000_000, dtype=jnp.float32)

# Demo only: jax.clear_caches() empties the in-process cache so a re-run shows the first-call compile cost again.
# Heads up: this does NOT clear the persistent on-disk cache enabled by JAX_COMPILATION_CACHE_DIR. If that environment
# variable is set (some managed environments turn it on by default), the "first call" will skip compilation and look fast.
jax.clear_caches()

t0 = time.perf_counter()
_ = heavy_jit(x).block_until_ready()
first_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
for _ in range(20):
    _ = heavy_jit(x).block_until_ready()
cached_ms = (time.perf_counter() - t0) * 1000 / 20

print(f"First call  (compile + execute): {first_ms:8.2f} ms")
print(f"Cached call (execute only):      {cached_ms:8.2f} ms")
print(f"Compilation cost (approx):       {first_ms - cached_ms:8.2f} ms")

The gap between the two numbers is roughly how much XLA spent compiling. For tiny functions it's a few tens of milliseconds; for a full transformer training step it can easily be several seconds. The good news is that you pay it **once per shape/dtype combination**, not per call. The rest of this lesson is about making sure you don't pay it more often than you have to.

## The compile cache and what triggers a recompile

JAX keys the compile cache on a **structural signature** of the inputs: their shapes, their dtypes, and any arguments marked as static. If the signature matches one JAX has seen before, the cached executable runs. If anything changes, JAX traces and compiles again.

Three things commonly trigger a recompile:

<div style="font-family: Arial, sans-serif; max-width: 980px; line-height: 1.35; display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 14px; align-items: stretch; margin-top: 4px;">
  <div style="border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">1. Shape</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Different shape</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;"><code>(32, 128)</code> and <code>(16, 128)</code> are separate cache entries.</div>
  </div>
  <div style="border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">2. Dtype</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Different dtype</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;"><code>float32</code> and <code>bfloat16</code> are separate cache entries.</div>
  </div>
  <div style="border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background: #ffffff;">
    <div style="font-size: 12px; color: #0969da; background: #ddf4ff; display: inline-block; padding: 2px 8px; border-radius: 12px;">3. Static arg</div>
    <div style="font-size: 15px; font-weight: 700; margin-top: 8px;">Different static value</div>
    <div style="font-size: 13px; color: #3b424a; margin-top: 6px;">The value of any <code>static_argnums</code> / <code>static_argnames</code> argument is part of the cache key (more on this below).</div>
  </div>
</div>
<br>

The **values** of regular array inputs do *not* matter. Two `(32, 128)` `float32` arrays with completely different contents hit the same compiled executable.

Let's watch a recompile happen.

In [ ]:
@jax.jit
def f(x):
    """Trivial jitted scalar function used to demonstrate one compile per new input shape (a new dtype would trigger the same recompile)."""
    return jnp.sum(x ** 2)

# Demo only: clear JAX's in-process compilation cache so rerunning this cell
# still shows slow first calls for new shapes.
jax.clear_caches()

# Feed in a few different shapes and measure each call.
shapes = [(100,), (200,), (100,), (200,), (300,)]
for s in shapes:
    x = jnp.ones(s, dtype=jnp.float32)
    t0 = time.perf_counter()
    _ = f(x).block_until_ready()
    dt = (time.perf_counter() - t0) * 1000
    print(f"shape={s!s:8s}  {dt:7.2f} ms")

You should see three slow calls (one per new shape) and two fast ones (shape `(100,)` and `(200,)` repeated). Real workloads do this accidentally all the time — variable-length sequences, the last batch in an epoch, ragged tokenization output. The fix in almost every case is *don't let the shape change*.

## Python control flow on a traced value

Tracing has one important consequence that catches new users every time. During tracing, your function's inputs aren't concrete arrays — they're abstract values with a known shape and dtype but no values. Any Python construct that needs to compare those values numerically (`if`, `while`, `bool(x)`, `int(x)`) breaks tracing.

Let's see what that looks like.

In [ ]:
@jax.jit
def relu_bad(x):
    """Wrong-by-design ReLU using a Python `if` on a traced value; JIT errors out at trace time."""
    if x > 0:
        return x
    return jnp.zeros_like(x)


try:
    print(relu_bad(jnp.array(1.0)))
except jax.errors.TracerBoolConversionError as e:
    print(f"{type(e).__name__}: {str(e).splitlines()[0]}")

The error message points at the offending `if`: JAX can't decide which branch to keep when the value is abstract. The fix is to express the choice as *data*, not Python control flow. For a small elementwise selection like ReLU, **`jnp.where`** is the cleanest tool — both branches always run, and the predicate tells JAX which one to use elementwise.

> **Gotcha:** because *both* sides of `jnp.where` are evaluated, expressions like `jnp.where(x > 0, jnp.log(x), 0.0)` still compute `log(x)` at `x <= 0` and propagate NaNs into the gradient. Guard the unsafe branch instead, e.g. `jnp.log(jnp.where(x > 0, x, 1.0))`.

In [ ]:
@jax.jit
def relu(x):
    """Correct ReLU using `jnp.where`; both branches are computed so tracing works."""
    return jnp.where(x > 0, x, 0.0)


print(relu(jnp.array([-1.0, -0.5, 0.0, 0.5, 1.0])))

For branches that compute very different things — where running both would be wasteful — use **`jax.lax.cond`**. Both branch functions are traced, but at runtime `lax.cond` represents an XLA conditional, so normally only the selected branch executes. One caveat: under `vmap`, `cond` may be converted to a select-like operation.

In [ ]:
@jax.jit
def soft_or_sharp(x, sharp):
    """Switch between hard ReLU and softplus inside the compiled graph via `lax.cond`, controlled by a traced bool."""
    # `sharp` is a scalar bool; lax.cond compiles to a real if-then-else
    return jax.lax.cond(
        sharp,
        lambda x: jnp.where(x > 0, x, 0.0),  # hard ReLU
        lambda x: jax.nn.softplus(x),         # smooth alternative
        x,
    )


x = jnp.array([-1.0, 0.5, 2.0])
print(f"sharp=True:  {soft_or_sharp(x, jnp.array(True))}")
print(f"sharp=False: {soft_or_sharp(x, jnp.array(False))}")

For loops over traced data, use structured control-flow primitives such as **`jax.lax.while_loop`**, **`jax.lax.fori_loop`**, and **`jax.lax.scan`**. 

Here is the practical difference. A Python `for` loop with a static bound is valid inside `jit`, but JAX unrolls the loop while tracing. That means 200 loop iterations become roughly 200 repeated blocks in the compiled program. `lax.scan` keeps the loop as a loop-like primitive, which usually compiles much faster for long fixed-length loops.

In [ ]:
# python_for_loop compile time scales with NUM_STEPS because the trace unrolls every iteration into the HLO;
# scan_loop compile time stays roughly constant. Try NUM_STEPS = 2000 to see the gap widen dramatically.
NUM_STEPS = 200


@jax.jit
def python_for_loop(x):
    """Python `for` loop inside jit; traced eagerly, so XLA sees one large unrolled program."""
    y = x
    for _ in range(NUM_STEPS):
        y = jnp.sin(y) + 0.01 * y
    return y


@jax.jit
def scan_loop(x):
    """Same logic expressed with `lax.scan`; traced once into a compact loop in the HLO."""
    def body(y, _):
        y = jnp.sin(y) + 0.01 * y
        return y, None

    y, _ = jax.lax.scan(body, x, xs=None, length=NUM_STEPS)
    return y


x = jnp.ones((1024,), dtype=jnp.float32)

jax.clear_caches()

t0 = time.perf_counter()
_ = python_for_loop(x).block_until_ready()
python_for_ms = (time.perf_counter() - t0) * 1000

jax.clear_caches()

t0 = time.perf_counter()
_ = scan_loop(x).block_until_ready()
scan_ms = (time.perf_counter() - t0) * 1000

print(f"Python for loop first call: {python_for_ms:8.2f} ms")
print(f"lax.scan first call:        {scan_ms:8.2f} ms")

Both functions compute the same recurrence. The important difference is compile-time structure: the Python loop is unrolled during tracing, while `lax.scan` lowers to a loop primitive. For short loops, a Python `for` loop is often fine. For long differentiable loops, `lax.scan` is usually the better default.

## Padding to stabilize shapes

Real workloads love to vary in shape — the last batch in an epoch is smaller, sequences have different lengths, KV caches grow over time. Every one of those triggers a fresh compile if you let the shape leak through to JAX. The standard fix is to **pad inputs to a fixed shape and mask the unused positions**.

In [ ]:
MAX_LEN = 16


@jax.jit
def masked_mean(x, mask):
    """Mean of `x` ignoring positions where `mask==0`; always called with shape (MAX_LEN,) so it compiles once."""
    # Always called with shape (MAX_LEN,) - no recompile when actual length varies
    return jnp.sum(x * mask) / jnp.maximum(jnp.sum(mask), 1.0)


def pad(seq):
    """Right-pad a variable-length list of floats to `MAX_LEN` and return the padded array plus a 0/1 mask."""
    actual_len = len(seq)
    if actual_len > MAX_LEN:
        raise ValueError(f"sequence length {actual_len} exceeds MAX_LEN={MAX_LEN}")

    pad_len = MAX_LEN - actual_len
    x = jnp.concatenate([
        jnp.asarray(seq, dtype=jnp.float32),
        jnp.zeros(pad_len, dtype=jnp.float32),
    ])
    mask = jnp.concatenate([
        jnp.ones(actual_len, dtype=jnp.float32),
        jnp.zeros(pad_len, dtype=jnp.float32),
    ])
    return x, mask


# Several different sequence lengths, but a single compiled function handles them all
for seq in [[1.0, 2.0, 3.0], [10.0] * 8, [5.0, -2.0]]:
    x, mask = pad(seq)
    print(f"len={len(seq):2d}  mean={masked_mean(x, mask):.3f}")

All three calls hit the same compiled executable because the shape on the device is always `(MAX_LEN,)`. Only the mask changes. This same pattern shows up at every scale — from a 16-element toy mean here, to padded attention masks in large-scale transformer training.

## When you *want* a recompile: `static_argnums`

Sometimes a parameter genuinely is a Python-side constant — a layer count, a precision flag, a kernel size — and you want JAX to bake its value into the compiled program. Mark those arguments with **`static_argnums`** (or **`static_argnames`** for keyword arguments). JAX will hash the *value* of those arguments into the cache key, so each distinct value gets its own compiled executable.

In [ ]:
@partial(jax.jit, static_argnums=0)
def power(n: int, x):
    """Repeated squaring; `n` is static so JAX unrolls the loop and compiles a fresh program per value of `n`."""
    # `n` is a Python int; JAX bakes it into the trace and unrolls the loop
    y = x
    for _ in range(n):
        y = y * y
    return y


# Demo only: clear JAX's in-process compilation cache so rerunning this cell
# still shows one compile per new static value.
jax.clear_caches()

for n in (2, 3, 2):  # n=2 reuses the cache the second time
    t0 = time.perf_counter()
    _ = power(n, jnp.arange(4, dtype=jnp.float32)).block_until_ready()
    print(f"n={n}: {(time.perf_counter() - t0) * 1000:7.2f} ms")

Each new value of `n` triggers a compile, but for fixed configurations that's exactly what you want — the loop unrolls completely and XLA can see every operation. The trade-off is straightforward: don't put a continuously-varying value in `static_argnums`, or you'll recompile on every call.

> **Tip:** A common static argument is a Python `bool` or `int` controlling a code path (training vs. inference, dropout on/off). Keep `static_argnums` reserved for things that change rarely — once per script run, not once per batch.

## Inspecting the trace

When something compiles differently than you expect, **`jax.make_jaxpr`** lets you see the trace before XLA touches it. It's a JAX-level compiler intermediate representation (IR): a typed, functional representation of what JAX staged out before lowering to StableHLO and then XLA. It is not the final optimized GPU code, but it is very useful for understanding what JAX traced.

In [ ]:
def f(x):
    """Same expression as the first demo, used here to print its `jaxpr` (JAX's intermediate representation)."""
    return jnp.tanh(x) * jnp.sin(x) + jnp.log1p(x * x)


print(jax.make_jaxpr(f)(jnp.arange(4, dtype=jnp.float32)))

Each line is a single primitive operation on typed arrays. If you ever suspect that JAX is recompiling because a shape or dtype changed unexpectedly, comparing two jaxprs from "fast" and "slow" calls usually pinpoints the culprit. The same trick works for figuring out why a transformation (`grad`, `vmap`) is producing more work than you intended.

## Summary

In this lesson, you built the mental model behind `jax.jit`: JAX traces your Python function, turns the traced computation into compiler input, and caches the compiled executable for matching input signatures. Most surprising JIT behavior becomes easier to explain once you ask: "Did I change the shape, dtype, static argument, or Python control flow?"

In this notebook you learned to:

* **Trace** a Python function with `jax.jit` and see that Python side effects run during tracing, not every execution.
* **Distinguish** compilation time from cached execution time with simple timing.
* **Identify** the common inputs to the compile cache: input PyTree structure, shapes, dtypes, and static arguments.
* **Avoid** traced-value control-flow errors by replacing Python branching on JAX values with `jnp.where` or `jax.lax` control flow.
* **Stabilize** input shapes with padding and masking so one compiled executable can handle a range of examples.
* **Bake** Python-side constants into the trace with `static_argnums` when you intentionally want a separate executable.
* **Inspect** the JAX-level trace with `jax.make_jaxpr` when a function compiles differently than you expect.

The high-order rule: keep shapes and dtypes stable, make static values intentional, and write data-dependent control flow in JAX rather than ordinary Python.